## Topic: Maximal Marginal Relevance (MMR) retriever

### Agenda
- 1. Introduction of Maximal Marginal Relevance (MMR) retriever

- 2. Syntax of MMR retriever

- 3. Practical Examples of MMR retriever

- 4. Complete Summary of MMR retriever

### 1. Introduction of Maximal Marginal Relevance (MMR) retriever

- Definition:
    - Maximal Marginal Relevance (MMR) is a retrieval strategy that balances relevance AND diversity.

    - MMR is an information retrieval algorithm designed to reduce redundancy in the retrieved results while maintaining high relevance to the query.


- Key Rule of thumb:
    - How can we pick results that are not only relevant to the query but also different from each other?

    - Give me chunks that are still relevant, but not all saying the same thing.



In [ ]:
""" 
Workflow
========

Query → Embed → Fetch many candidates (fetch_k)
                    ↓
              MMR re-selects k diverse docs
                    ↓
              Prompt → LLM

"""

In [ ]:
"""     how to work internally?
    ==================================
    
┌─────────────────────────────────────────────────────────────┐
│                 MMR RETRIEVER INTERNAL FLOW                 │
│                                                             │
│  1. EMBED THE QUERY                                         │
│     q → embedding vector                                    │
│                                                             │
│  2. FETCH A LARGE CANDIDATE POOL                            │
│     similarity_search with fetch_k (e.g. 20)                │
│     Not the final k — a bigger pool to pick from            │
│                                                             │
│  3. ITERATIVE MMR SELECTION                                 │
│     Selected = []                                           │
│     Repeat until |Selected| == k:                           │
│                                                             │
│       For each remaining candidate d:                       │
│         Relevance  = sim(q, d)                              │
│         Redundancy = max sim(d, s) for s in Selected        │
│         MMR_score  = λ * Relevance − (1−λ) * Redundancy     │
│                                                             │
│       Pick d with highest MMR_score → add to Selected       │
│                                                             │
│  4. RETURN List[Document] of size k                         │
│     Diverse, still query-relevant chunks                    │
└─────────────────────────────────────────────────────────────┘

- Key Rule of Thumb:
    First pick: always the most similar doc (no selected set yet).
    Later picks: penalize docs too similar to what you already have

"""

In [ ]:
""" 
Why MMR Retriever?
====================

Why MMR Retriever?

- Problems with plain similarity:
    - Redundant chunks — same idea, 4 slightly different splits
    - Lost coverage — one section dominates; other relevant sections never appear
    - Wasted tokens — LLM reads the same fact four times
    - Biased answers — one viewpoint / one paragraph overweighted


- MMR helps when:
    - Long PDFs with overlapping splits (chunk_overlap)
    - FAQs with many similar Q&A pairs
    - Product catalogs (don't return 5 variants of the same SKU description)
    - Multi-aspect questions (“pros, cons, and history of X”)
    - You want broader context in a small k

- When not to prefer MMR:
    - You need the single most similar passage (citation of one clause)
    - Corpus is already highly unique (little duplication)
    - fetch_k is too small — diversity has nothing to choose from

"""

### 2. Syntax of MMR retriever

In [ ]:
"""  Syntax of MMR retriever
  ===================================

retriever = vectorstore.as_retriever(
    search_type="mmr",       # ← THE KEY PARAMETER
    search_kwargs={
        "k": 4,              # final number of docs
        "fetch_k": 20,       # candidate pool (must be > k)
        "lambda_mult": 0.5,  # 1=relevance, 0=diversity
        
        "filter": {"source": "handbook.pdf"},  # optional
    },
)

docs = retriever.invoke("What is self-attention?")

"""

""" 

================================***=====================================


┌─────────────────────────────────────────────────────────────┐
│              IMPORTANT RETRIEVER PARAMETERS                 │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ CORE PARAMETERS (All Retrievers)                      │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ k                  → Number of docs to return         │  │
│  │                      Default: 4                       │  │
│  │                      Range: 1-100+                    │  │
│  │                                                       │  │
│  │ search_type        → Retrieval strategy               │  │
│  │                      "similarity" (default)           │  │
│  │                      "mmr" (diverse results)          │  │
│  │                      "similarity_score_threshold"     │  │
│  └───────────────────────────────────────────────────────┘  │
│                                                             │
│  ┌───────────────────────────────────────────────────────┐  │
│  │ SEARCH_KWARGS (Vector Store Retrievers)               │  │
│  ├───────────────────────────────────────────────────────┤  │
│  │ fetch_k            → Candidate pool before MMR        │  │
│  │                      Default: 20                      │  │
│  │                      Use: fetch_k >> k (e.g., 20 vs 4)│  │
│  │                                                       │  │
│  │ lambda_mult        → MMR diversity factor             │  │
│  │                      Range: 0.0 - 1.0                 │  │
│  │                      0.0 = Maximum diversity          │  │
│  │                      1.0 = Maximum relevance          │  │
│  │                      Default: 0.5                     │  │
│  │                                                       │  │
│  │ score_threshold    → Minimum similarity score         │  │
│  │                      Range: 0.0 - 1.0                 │  │
│  │                      Use: Filter out weak matches     │  │
│  │                      Example: 0.7 = only strong matches│ │
│  │                                                       │  │
│  │ filter             → Metadata filter (pre-search)     │  │
│  │                      {"year": {"$gte": 2020}}         │  │
│  │                      {"category": "tech"}             │  │
│  └───────────────────────────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────┘


"""

### 3. Practical Examples of MMR retriever

In [1]:
# Example 1: MMR

# import necessary library
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

c:\Users\kz\anaconda3\envs\GenAI\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.6.0)/charset_normalizer (3.5.1) doesn't match a supported version!
  warnings.warn(


In [10]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [11]:
# Step 2: Initialize the Embedding model
# Download from the Huggingface platform then initialize the model 

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1781.63it/s]


In [12]:
# Step 3: Create a FAISS Vector Database
vectorstore = FAISS.from_documents(
    documents = docs,
    embedding= embedding_model
    
)

In [19]:
# Step 4: Convert vectorstore into a MMR retriever
retriever = vectorstore.as_retriever(
    search_type = "mmr", 
    search_kwargs = {
        "k": 3,
        "lambda_mult": 0.5  # 0.0 = Maximum diversity, 1.0 = Maximum relevance 
    }

)

In [20]:
query = "What is langchain?"
results = retriever.invoke(query)

In [21]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
LangChain supports Chroma, FAISS, Pinecone, and more.

--- Result 2 ---
LangChain is used to build LLM based applications.

--- Result 3 ---
Embeddings are vector representations of text.


### Example 2: Full RAG Pipeline with MMR

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# Retriever with MMR
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5}
)

prompt = ChatPromptTemplate.from_template("""
Answer using ONLY the context below. Give a comprehensive answer
covering different aspects mentioned in the context.

Context:
{context}

Question: {question}
""")

llm = ChatOpenAI(model="gpt-4o-mini")

def format_docs(docs):
    return "\n\n".join(f"- {d.page_content}" for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("Compare flagship smartphones of 2024")
print(answer)

# Because MMR retrieved iPhone + Samsung + Pixel (not 3x iPhone),
# the LLM can actually COMPARE — impossible with similarity results!

###  4. Complete Summary of MMR retriever

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│                  MMR (MAXIMAL MARGINAL RELEVANCE)                │
│                                                                  │
│  WHAT:  Retrieval that maximizes relevance while minimizing      │
│         redundancy — returns diverse + relevant documents        │
│                                                                  │
│  WHY:   Similarity returns near-duplicates (wasted context).     │
│         MMR gives 3x information density for same k.             │
│                                                                  │
│  FORMULA:                                                        │
│    MMR = λ * Sim(doc, query) - (1-λ) * max Sim(doc, selected)    │
│                                                                  │
│  SYNTAX (Retriever):                                             │
│    retriever = vectorstore.as_retriever(                         │
│        search_type="mmr",                                        │
│        search_kwargs={                                           │
│            "k": 4,              # final docs                     │
│            "fetch_k": 20,       # candidate pool (5x k)          │
│            "lambda_mult": 0.5   # 1=relevant, 0=diverse          │
│        }                                                         │
│    )                                                             │
│                                                                  │
│  SYNTAX (Direct):                                                │
│    vectorstore.max_marginal_relevance_search(                    │
│        query, k=4, fetch_k=20, lambda_mult=0.5)                  │
│                                                                  │
│  KEY PARAMS:                                                     │
│    k           → Final results (default: 4)                      │
│    fetch_k     → Pool before MMR (MUST be > k, use 5-10x)        │
│    lambda_mult → 0.0-1.0 tradeoff knob                           │
│                  1.0 = similarity, 0.5 = balanced, 0.0 = diverse │
│                                                                  │
│  LAMBDA GUIDE:                                                   │
│    1.0 → Exact lookup      0.7 → Precise Q&A                     │
│    0.5 → Balanced default  0.3 → Recommendations                 │
│    0.0 → Avoid (irrelevant risk)                                 │
│                                                                  │
│  STRENGTHS:                                                      │
│    ✅ Eliminates duplicate/overlapping chunks                    │
│    ✅ Higher information density per token                       │
│    ✅ Better for summary, compare, overview queries              │
│    ✅ Safety net for imperfect chunking                          │
│                                                                  │
│  WEAKNESSES:                                                     │
│    ❌ Slightly slower than similarity                            │
│    ❌ Extra params to tune (fetch_k, lambda)                     │
│    ❌ Can diversify AWAY from single correct answer              │
│    ❌ Useless if fetch_k == k or k == 1                          │
│                                                                  │
│  WHEN TO USE vs ALTERNATIVES:                                    │
│    mmr                         → Default for RAG (diverse)       │
│    similarity                  → Exact fact, k=1, max speed      │
│    similarity_score_threshold  → Need "I don't know" behavior    │
│    Ensemble (BM25 + MMR)       → Production hybrid search        │
│                                                                  │
│  PIPELINE:                                                       │
│    Query → [Embed] → [Fetch 20 by similarity]                    │
│                        ↓                                         │
│                   [MMR greedy select 4]                          │
│                        THIS COMPONENT                            │
│                        ↓                                         │
│                   Top-4 diverse → [LLM]                          │
│                                                                  │
│  GOLDEN RULE:                                                    │
│  "Use MMR as your default (k=4, fetch_k=20, λ=0.5).              │
│   If answers feel unfocused, raise λ to 0.7.                     │
│   If answers feel repetitive, lower λ to 0.3."                   │
└──────────────────────────────────────────────────────────────────┘


"""